


##  Introduction:
In this task, we design a recommender system for Steam platform users using Steam data. The main goal was to use users' behavioral data, especially purchases and game duration, to generate recommendations for games according to their personal tastes. To achieve this goal, the process of generating the proposed system required several steps, including  Data Pre-processing, Exploratory Data Analysis, Model Training, Model Evaluation, Hyperparameter Tuning, and Recommender Generation.

This implementation was developed using the Alternating Least Squares (ALS) algorithm, as it is well suited for collaborative filtering problems .

One of the main considerations in this work was the structure of the dataset itself. The original data did not contain user ratings, meaning a rating signal had to be extracted from user behavior, so the hours of play were converted into feedback scores and used as input to the model. This was a key step because the quality of the recommendation model directly depended on how well the user’s preferences were represented before the model was trained.
   


###  


##  1 ) Data Pre-processing
##
####- Experiment Setup and Mlflow Tracking:
###
First, an MLflow test environment is configured to allow tracking of model execution, metrics, and evaluation criteria during the run.

MLflow was started to track the experiment and a test named  "```play_recommender```" was created or loaded if it existed.

Since the notebook was run in Databricks and its path might not be available during some runs, a **try/except block** was added to get the notebook's name. This helps avoid errors if the path is missing. The statement **split (/)[-1]** was used to grab the notebook name by picking the last part of the path.
###


In [ ]:
import mlflow
import logging

# Suppress unnecessary MLflow logs for cleaner output
logging.getLogger("mlflow").setLevel(logging.ERROR)
# Enable automatic tracking of model runs, parameters and metrics
mlflow.pyspark.ml.autolog()

# Access current notebook information
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
try:
  # Retrieve notebook path and extract notebook name
  n_path =ctx.notebookpath().get()
  nb = n_path.split("/")[-1]
except:
  # Use a default name if notebook metadata cannot be retrieved
  nb = "play"


# Configure Databricks as tracking destination
mlflow.set_tracking_uri("databricks")

# Configure model registry location
mlflow.set_registry_uri("databricks-uc")


# Define experiment path
experiment_name = f"/Shared/{nb}_recommender"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name)
# Set active experiment
mlflow.set_experiment(experiment_name)

print(" Experiment set to:",experiment_name)

 Experiment set to: /Shared/play_recommender


####- Data Import and Initial Data Loading:
###
The goal of this step was to import the Steam data into the Spark environment and convert it into a DataFrame, as data's structure should be checked before any kind of analysis.

checked the file structure and found that there wasn't an actual header. To avoid misreading the first row as a column name, we set   "```header=False```".

We also used "```InferSchema=True```" to allow Spark to automatically identify the data types of the columns.



In [ ]:
# Specify the dataset location
path = "PATH_TO_UNIVERSITY_DATASET"

# Read the dataset into Spark
steamdf = (
     spark.read
     .option("header" ,"false")
     .option("inferSchema" , "true")
     .csv(path)
     )
# Display sample records
steamdf.show (10, truncate=False)


+---------+--------------------------+--------+-----+
|_c0      |_c1                       |_c2     |_c3  |
+---------+--------------------------+--------+-----+
|151603712|The Elder Scrolls V Skyrim|purchase|1.0  |
|151603712|The Elder Scrolls V Skyrim|play    |273.0|
|151603712|Fallout 4                 |purchase|1.0  |
|151603712|Fallout 4                 |play    |87.0 |
|151603712|Spore                     |purchase|1.0  |
|151603712|Spore                     |play    |14.9 |
|151603712|Fallout New Vegas         |purchase|1.0  |
|151603712|Fallout New Vegas         |play    |12.1 |
|151603712|Left 4 Dead 2             |purchase|1.0  |
|151603712|Left 4 Dead 2             |play    |8.9  |
+---------+--------------------------+--------+-----+
only showing top 10 rows


####- Column Standardization :
###
In this part, the raw columns were converted into a meaningful schema that it could be used for analysis and modeling.

Since the initial data was without a header and the columns were displayed as
 _c0, _c1, _c2, _c3, This step was performed to convert the name of rawa into an understandable structure suitable for continuing the pipeline.
 then the column's names were changed to **user, game, behavior, and value.** This ensured that each column had a clear meaning.
This change was not only for readability, but also necessary for later steps such as filtering, grouping, joining, and defining the user-item matrix in the ALS model.

The data structure was also checked using **printSchema( )** to validate the data type of each column and ensure that the created schema was compatible with the needs of the next steps.


In [ ]:
# Replace default column names with more descriptive labels
steamdf = (

    steamdf
    .withColumnRenamed("_c0","user")
    .withColumnRenamed("_c1","game")
    .withColumnRenamed("_c2","behavior")
    .withColumnRenamed("_c3","value")
)

# Preview updated dataset
steamdf.show (10, truncate=False)

# Verify updated schema
steamdf.printSchema()


+---------+--------------------------+--------+-----+
|user     |game                      |behavior|value|
+---------+--------------------------+--------+-----+
|151603712|The Elder Scrolls V Skyrim|purchase|1.0  |
|151603712|The Elder Scrolls V Skyrim|play    |273.0|
|151603712|Fallout 4                 |purchase|1.0  |
|151603712|Fallout 4                 |play    |87.0 |
|151603712|Spore                     |purchase|1.0  |
|151603712|Spore                     |play    |14.9 |
|151603712|Fallout New Vegas         |purchase|1.0  |
|151603712|Fallout New Vegas         |play    |12.1 |
|151603712|Left 4 Dead 2             |purchase|1.0  |
|151603712|Left 4 Dead 2             |play    |8.9  |
+---------+--------------------------+--------+-----+
only showing top 10 rows
root
 |-- user: integer (nullable = true)
 |-- game: string (nullable = true)
 |-- behavior: string (nullable = true)
 |-- value: double (nullable = true)



## 2 ) Exploratory Data Analysis
##
####- Behavioral Filtering and Interaction Analysis:
###
This section deals with the quality of behavioral data and the analysis of the distribution of user interactions before entering into modeling.
Since the entire recommendation system in this problem is built based on user behavior, it was first necessary to determine what structure the behaviors in the data have, how they are distributed.
This step was also necessary to determine which type of behavior (purchase or play) would be a more appropriate basis for building a preference signal.

At this stage, rows with null values in the behavior column were removed by **dropna()**, because null data could distort the analyses. Then, the data was grouped and was counted based on the behavior column using **groupBy() and count()** to calculate the frequency of different user behaviors. This analysis was performed to compare the distribution of two main behaviors, namely purchase and play.

In [ ]:
# Exclude rows where behaviour information is missing
steamdf=steamdf.dropna(subset=["behavior"])

# Summarise how often each behaviour appears in the dataset
steamdf.groupBy("behavior").count().orderBy("count", ascending=False).display()


behavior,count
purchase,129511
play,70489


####- Output :
###
At this point, it is clear that both shopping and gaming behaviors are present. However, play behavior was considered a stronger signal of user preference because it reflects actual engagement with a game, while purchase behavior only shows ownership and does not necessarily indicate sustained interest.
###

####- purchase popularity vs. play engagement analysis :
###
At this point, understanding the behavioral pattern is essential, an exploratory analysis is performed to examine the difference between popularity based on purchases and engagement based on hours played.

The goal was to determine whether the games that had the most purchases were necessarily the ones that generated the most real users.


First, the purchase records were filtered and the games were grouped and segmented based on the number of purchases.
Then, for playback, instead of simply counting the records, the sum value (which represents the game) was calculated:

- Purchase popularity → Number of purchases
- Engagement → Total hours played ```sum("value")```


In [ ]:
import pyspark.sql.functions as F
# Find games with the highest number of purchases
top_purchase =(
     steamdf.filter(F.col("behavior") == "purchase")
     .groupBy("game")
     .count()
     .orderBy(F.desc("count"))
     .limit(10)

     )
# Find games generating the highest total playtime
top_play =(
     steamdf.filter(F.col("behavior") == "play")
     .groupBy("game")
     .agg(F.sum("value").alias("total_hours"), F.avg("value").alias("avg_hours"))
     .orderBy(F.desc("total_hours"))
     .limit(10)
     )

display(top_purchase)
display(top_play)




game,count
Dota 2,4841
Team Fortress 2,2323
Unturned,1563
Counter-Strike Global Offensive,1412
Half-Life 2 Lost Coast,981
Counter-Strike Source,978
Left 4 Dead 2,951
Counter-Strike,856
Warframe,847
Half-Life 2 Deathmatch,823


game,total_hours,avg_hours
Dota 2,981684.5999999999,202.78549886387108
Counter-Strike Global Offensive,322771.6000000001,234.40203340595505
Team Fortress 2,173673.30000000005,74.7625053809729
Counter-Strike,134261.09999999998,236.37517605633798
Sid Meier's Civilization V,99821.30000000002,180.1828519855596
Counter-Strike Source,96075.50000000003,134.3713286713287
The Elder Scrolls V Skyrim,70889.3,104.71093057607091
Garry's Mod,49725.3,74.66261261261262
Call of Duty Modern Warfare 2 - Multiplayer,42009.899999999994,144.3639175257732
Left 4 Dead 2,33596.700000000004,41.9434456928839


#### - Output :
The output of this step produced two different rankings:
- Top-selling games
- Most-engaged games

Comparing the two showed that popularity is not necessarily the same as buying and engaging. Games may have more purchases, but generate fewer play hours.

####- Generating game IDs :
###
Before training the ALS model, it was important to change the game names from text into numbers. This is because the ALS algorithm only works with numeric IDs for users and items to create the user-item matrix and cannot use the actual game names.

To tackle this issue:
- the column that contained the game names was extracted by using **select("game")**.
- Next, any duplicate games with **distinct()** was removed so that each game appeared only once in our table.
- In this process, all unique game names were sorted alphabetically using "```Window.orderBy("game")```".
- **row_number()** assigned a sequential number to each game based on that sort order.
- The **over()** function simply attached row_number() to the defined window to apply the numbering to all the sorted game records.

Thus, unique, consistent, and appropriate identifiers were created to create the user-item interaction matrix in the ALS recommendation model.

In [ ]:
from pyspark.sql.window import Window
gamedf = (

    # Select only game titles from the dataset
    steamdf.select("game")

     # Keep unique game names only
    .distinct()

    # Arrange game titles in a fixed order
    .orderBy("game")

    # Create sequential numeric IDs for each game
    .withColumn("game_id",F.row_number().over(Window.orderBy("game")))
)
display(gamedf.limit(10))

game,game_id
007 Legends,1
0RBITALIS,2
1... 2... 3... KICK IT! (Drop That Beat Like an Ugly Baby),3
10 Second Ninja,4
"10,000,000",5
100% Orange Juice,6
1000 Amps,7
12 Labours of Hercules,8
12 Labours of Hercules II The Cretan Bull,9
12 Labours of Hercules III Girl Power,10


#### - Constructing implicit ratings from play hours:
###
Raw play hours cannot be input into the model without processing, because game hours are so scattered. Some users play a few hours, some hundreds.

If this were fed directly into the model, users with very high hours could bias the model.

Initially, only game behavior was kept, as it reflects the user's actual preferences better than purchases.

Then:
- The value column was converted to a numeric type (double) and renamed hours.
- Invalid or zero values ​​were removed.
- The rating was then constructed as follows: **rating = log1p(hours)** --->
 ( That means:    rating =log(1 + hours) )

The logarithmic transformation was used to:
- Control the effect of very large values
- Make the data distribution more balanced
- Make the data more stable for ALS

In [ ]:
playdf = (

    steamdf.filter(F.col("behavior") == "play")
    # Merge play records with generated game IDs
    .join(gamedf, ["game"])
    # Convert columns to proper numeric formats
    .withColumn("game_id", F.col("game_id").cast("int"))
    .withColumn("user", F.col("user").cast("int"))
    # Exclude invalid play records
    .filter(F.col("value") > 0)
    # Store play duration as hours
    .withColumn("hours", F.col("value").cast("double"))
    # Convert hours into implicit ratings
    .withColumn("rating", F.log1p("hours"))

)
# Select final columns
playdf =(
    playdf.select("game_id","game","user","hours","rating")
)

# Display top play records
display(playdf.orderBy(F.desc("hours")).limit(10))
# Review schema structure
playdf.printSchema()


game_id,game,user,hours,rating
3826,Sid Meier's Civilization V,73017395,11754.0,9.37203396097417
1337,Dota 2,100630947,10442.0,9.253687176479549
4258,Team Fortress 2,153382649,9640.0,9.17378011666438
1337,Dota 2,130882834,7765.0,8.957510510291613
1337,Dota 2,52567955,6964.0,8.848652886213905
1337,Dota 2,121199670,6753.0,8.817890200945513
1337,Dota 2,86256882,6015.0,8.702177865629675
3826,Sid Meier's Civilization V,70487610,6013.0,8.701845363548474
1337,Dota 2,101414179,5982.0,8.696677393390031
1337,Dota 2,12660489,5970.0,8.694669696546992


root
 |-- game_id: integer (nullable = false)
 |-- game: string (nullable = true)
 |-- user: integer (nullable = true)
 |-- hours: double (nullable = true)
 |-- rating: double (nullable = true)



####- Rating Range Validation :
###
At this point, the minimum and maximum values  of the score column indicate that the logarithmic transformation was performed correctly.

 Since the ALS model is trained based on rating and not raw game hours, it was necessary to validate the range of these values  before entering the model training phase.
Meth
The minimum and maximum values  of the rating column were calculated using **selectExpr()**:
- Minimum value of rating
- Maximum value of rating
###

In [ ]:
# Review rating boundaries
playdf.selectExpr("min(rating) as min_rating ", "max(rating) as max_rating").show()


+-------------------+----------------+
|         min_rating|      max_rating|
+-------------------+----------------+
|0.09531017980432487|9.37203396097417|
+-------------------+----------------+



####- Output :
###
The output showed that the rating values ​​were in a more balanced and compressed range compared to the raw game hours. This shows that the logarithmic transformation reduced the impact of very large values ​​and made the data more stable for the ALS model.
###

## 3 ) Model Training and Evaluation
##
####- Train/Test Split for Validation:
###
In this phase, the split was done into two training and testing parts so that the model only looked at a part of the training and then on the data that was previously evaluated.
This step was needed to avoid optimistic evaluation (optimistic evaluation).
In this phase, the ratio was:
- 80% training
- 20% testing

The training part was used for the interaction pattern in the ALS model, and the test part was kept for the prediction error calculation and evaluation of RMSE.
Also, the number of rows of both was checked and the sum of train_rows and test_rows was compared with the total number of rows of play interactions to control the accuracy of the split.
###

In [ ]:
# Divide dataset into train and test sets
(training,test) = playdf.randomSplit([0.8,0.2], seed=100)

# Check split sizes
print("train rows:" , training.count())
print("test rows:" , test.count())


train rows: 56405
test rows: 14084


####- Output :
###
The output shows:
The sum of test-rows and train-rows is equal with the number of play records in the ninth cell, which indicates that:
- The data is partitioned without loss
- The partitioning was done correctly
- The test is ready to measure the real performance of the model

####- Baseline ALS Model training:
###
In this section, the ALS algorithm was used because it is one of the common methods in collaborative filtering.
In this cell:
- The user column as the user ID
- The game_id column as the game ID
- the rating column (previously constructed by logarithmic transformation of game hours) as the user preference signal were entered to the model.

The model was trained using training data and then applied to test data to produce prediction values.
To evaluate the model performance, the RMSE metric was used, which measures the difference between the actual (rating) and predicted values.

As shown in the output of this step, the model was able to produce prediction values  for the test data and the RMSE value was calculated as the model error (about 1.5 in this task)

This value indicates the level of error of the model in the initial state, that is, when no optimization has been done on the model parameters. Considering that the rating values  are on a logarithmic scale, this error value is expected and reasonable.
 finally , the ALS model extracted a general pattern of user behavior, but since we did not provide numerical values for the parameters, it used the default values.
###  

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator



# Configure ALS model
als = ALS(
    userCol="user",
    itemCol="game_id",
    ratingCol="rating",
    seed=100
    )

# Set evaluation metric
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
    )

# Train the baseline model
model = als.fit(training)

# Generate test predictions
prediction = model.transform(test).dropna()

# Display prediction results
prediction.limit(10).display()


# Measure baseline error
rmse_baseline = evaluator.evaluate(prediction)
print("BASELINE RMSE is:",rmse_baseline)



game_id,game,user,hours,rating,prediction
3,1... 2... 3... KICK IT! (Drop That Beat Like an Ugly Baby),65117175,1.2,0.7884573603642702,1.6137671
6,100% Orange Juice,197328486,35.0,3.58351893845611,0.29793268
8,12 Labours of Hercules,73835640,6.0,1.9459101490553132,2.019622
9,12 Labours of Hercules II The Cretan Bull,147602462,14.7,2.7536607123542622,1.873611
11,140,30246419,1.2,0.7884573603642702,0.26257622
14,16bit Trader,92914917,4.2,1.6486586255873816,1.1797504
26,3DMark,1950243,26.0,3.295836866004329,0.36612317
26,3DMark,3449240,0.3,0.26236426446749106,1.3879997
26,3DMark,9823354,0.3,0.26236426446749106,1.4414209
26,3DMark,11978743,170.0,5.14166355650266,1.4158653


BASELINE RMSE is: 1.5066419770783717


### 4) Hyperparameter Tuning (Optimized ALS) :
### ##
The goal of this step is to improve the performance of the ALS model, and for this, a network of ALS key parameters was defined:
- regParam to prevent overfitting
- maxIter for the convergence rate in training

TrainValidationSplit was used to:
- Split the training into two train/validation parts
- For each combination, a separate model was trained
The performance of each model depends on the RMSE value.
TrainValidationSplit Was used instead of manual testing.

Finally, the model with the lowest error was used as a best model for the test data.


In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# Define tuning combinations
parameters =(
     ParamGridBuilder()
    .addGrid(als.regParam, [ 0.2,0.1])
    .addGrid(als.maxIter, [10,20])
    .build()
    )

# Run train-validation tuning
tvs =(
     TrainValidationSplit(
        estimator=als,
        evaluator=evaluator,
        estimatorParamMaps=parameters,
        trainRatio=0.75,
        seed=100
    ))

# Train and retrieve best model
tvsModel = tvs.fit(training)
bestModel = tvsModel.bestModel

# Evaluate tuned model
prediction = bestModel.transform(test).dropna()
rmse = evaluator.evaluate(prediction)

# Store final metric
mlflow.log_metric("rmse",rmse)

# Print best settings
print("ERROR is: " + str(rmse))
print("Best MaxIter is:" , bestModel._java_obj.parent().getMaxIter())
print("Best getRegParam is:" , bestModel._java_obj.parent().getRegParam())


ERROR is: 1.4280947575303267
Best MaxIter is: 10
Best getRegParam is: 0.2


####- Output:
###
As shown in the output:
- The RMSE value of the optimized model (about 1.42) was calculated
- The best parameter values  were extracted

Comparison with the baseline (about 1.5) shows that the model has improved, although this improvement seems small on a numerical scale
although This RMSE reduction is limited, it is significant for the following reasons:
The rating value is on a logarithmic scale, so even small changes in RMSE are significant



## 5) Recommendations Generation
##
#### - generating recommendations for all users :
###
the optimized model was used to generate personalized recommendations for all users.
In this step, the function **recommendForAllUsers(10)**
was used, which generates the top 10 recommended items for each user based on the predicted score.

The output of this function includes:
- The user ID (user)
- A list of recommendations

where each item in this list contains:
- game_id
- rating (the score predicted by the model)

In [ ]:
# Generate ten name of top recommendations for every user
userRecs = bestModel.recommendForAllUsers(10)

# Show recommendation output
userRecs.select("user", slice("recommendations", 1, 5).alias("recommendations")).limit(10).display()


user,recommendations
948368,"List(List(985, 4.1018786), List(1607, 3.6318758), List(3046, 3.3210847), List(2152, 3.2389007), List(982, 2.725993), List(3045, 2.5818412), List(2941, 2.5752628), List(2863, 2.5206866), List(314, 2.46313), List(321, 2.3602288))"
975449,"List(List(178, 3.4002626), List(4365, 3.1978161), List(1607, 3.1611493), List(2582, 3.119361), List(2851, 3.102197), List(1446, 3.0070233), List(537, 2.9831693), List(563, 2.9816325), List(437, 2.9429688), List(2925, 2.9273932))"
2753525,"List(List(982, 4.3210735), List(2852, 3.9786997), List(4131, 3.891385), List(5042, 3.7878475), List(2851, 3.737093), List(1796, 3.6639328), List(3046, 3.6109722), List(178, 3.564522), List(3571, 3.4820325), List(3272, 3.4237))"
3450426,"List(List(3046, 1.5904578), List(3045, 1.242126), List(2658, 1.1073451), List(537, 1.0754436), List(1702, 1.0732238), List(3491, 1.0537256), List(4524, 1.036557), List(1067, 1.0169855), List(1703, 0.98697376), List(1979, 0.98104894))"
8567888,"List(List(3046, 0.84346664), List(985, 0.8061659), List(1607, 0.76507646), List(982, 0.67159426), List(3045, 0.6674158), List(2152, 0.63537395), List(3706, 0.5840222), List(1796, 0.5735312), List(738, 0.5709124), List(732, 0.5644662))"
8784496,"List(List(1621, 4.1190076), List(3046, 3.8380406), List(1679, 3.4897878), List(2925, 3.443005), List(1607, 3.3851626), List(3272, 3.3510644), List(4524, 3.3254309), List(437, 3.3231227), List(2852, 3.2644162), List(1702, 3.1456614))"
8795607,"List(List(3046, 3.1593935), List(178, 2.7293324), List(1621, 2.5470092), List(4524, 2.544271), List(1979, 2.5011103), List(2658, 2.4971864), List(620, 2.440284), List(1679, 2.4245062), List(3571, 2.42299), List(2852, 2.421854))"
10595342,"List(List(2658, 4.804376), List(3823, 4.6133823), List(1459, 4.0873775), List(3046, 3.886852), List(178, 3.8821561), List(1763, 3.8397114), List(3491, 3.7381473), List(2729, 3.6725173), List(2582, 3.6075408), List(4489, 3.5755057))"
10599862,"List(List(537, 4.953967), List(4668, 4.8387637), List(2851, 4.830378), List(2582, 4.7586184), List(2852, 4.736642), List(1761, 4.6504493), List(178, 4.6098456), List(1766, 4.4798903), List(4365, 4.416252), List(2850, 4.4001565))"
11973378,"List(List(3046, 0.5952372), List(985, 0.5689139), List(1607, 0.539917), List(982, 0.47394624), List(3045, 0.47099748), List(2152, 0.44838542), List(3706, 0.4121464), List(1796, 0.4047429), List(738, 0.40289477), List(732, 0.39834565))"


####- Output:
###
In the previous cell, you can see that the output is in the form of an array that the model considers for each user. At this stage, the output is presented in the form of game id , in the next step, this id will change to the actual names of the games.
###

####- Interpreting recommendations for a specific user:
###
At this point the raw output of the model (which only contained the game ID) was converted into a human-readable output.

Several important steps were performed:
- Filtering a specific user for more detailed analysis
"```(userRecs.user == 151603712)```"
- Since the suggestions were stored as an array, each suggestion item was converted into a separate row using **explode()** so that it could be analyzed
- The data was **joined** with the gamedf table to convert the **game_id** to the actual game name

In [ ]:
from pyspark.sql.functions import explode
(
# Select recommendations for a specific user
userRecs.where(userRecs.user == 151603712).select("recommendations")

# Convert recommendation array into individual rows
.withColumn("recommendations", explode("recommendations"))

# Extract recommended game IDs and predicted scores
.select("recommendations.game_id", "recommendations.rating")

# Match game IDs with actual game titles
.join(gamedf ,["game_id"] )

# Display final recommendation list
.show(truncate=False)
)

+-------+---------+-----------------------------------+
|game_id|rating   |game                               |
+-------+---------+-----------------------------------+
|537    |2.3767872|Black Ink                          |
|1607   |2.1467118|FIFA Manager 09                    |
|1700   |2.012758 |Farming Simulator 15               |
|2393   |2.0523589|King's Bounty Warriors of the North|
|2630   |2.0442216|Majesty 2 Collection               |
|2732   |2.1030233|Might & Magic Heroes VI            |
|2850   |2.305105 |NBA 2K13                           |
|4365   |1.9573114|The Elder Scrolls V Skyrim         |
|4668   |1.9855838|Total War ROME II - Emperor Edition|
|4669   |2.0017514|Total War SHOGUN 2                 |
+-------+---------+-----------------------------------+



####- Output:

In the final output, the user was presented with:
- A list of games (with real names)
- Along with the predicted score

This transformed the model output from a numerical and incomprehensible state into a usable list for the user

## Conclusion :

In this task, a complete recommendation workflow was developed using PySpark and ALS collaborative filtering. The data was first cleaned and transformed, and play behaviour was selected as the main interaction signal because it better reflects actual user engagement than purchase activity.

After exploratory analysis and rating normalization, a baseline ALS model was trained and then improved through hyperparameter tuning to reduce prediction error.

After exploratory analysis and rating normalization, a baseline ALS model was trained and then improved through hyperparameter tuning to reduce prediction error. The best model was able to generate games recommendations for users.

Overall, this task showed how user behavioural data can be changed into effective recommendation outputs through data preprocessing, model training, optimisation, and evaluation.